# Ders 2A — RF ile Feature Ablation

RandomForest sabit tutulur; farklı feature setleri karşılaştırılır.

## 1. Paket kontrolü

Bu scriptte gerçekten kullanılacak paketler kontrol edilir. Gereksiz paket import edilmez.

In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.

required_packages = [
    ("pandas", "pandas"),  # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),  # Sayısal matris ve vektör işlemleri için kullanılır.
    ("matplotlib", "matplotlib"),  # Grafik çizimleri için kullanılır.
    ("sklearn", "scikit-learn"),  # Makine öğrenmesi modelleri ve metrikleri için kullanılır.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.

## 2. Importlar ve genel ayarlar

Bu hücrede yalnızca bu scriptte kullanılacak paketler ve sabitler çağırılır.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Feature matrisleri ve sayısal işlemler için kullanılır.
import pandas as pd
# Feature CSV dosyalarını okumak ve sonuç tablolarını kaydetmek için kullanılır.
import matplotlib.pyplot as plt
# Feature ablation bar chartlarını çizmek için kullanılır.

from sklearn.model_selection import train_test_split
# Train/test ayrımı yapmak için kullanılır.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
# Feature ablation sonuçlarını ölçmek için gerekli metrikler çağırılır.
from sklearn.ensemble import RandomForestClassifier
# Feature set karşılaştırmasında sabit gatekeeper model olarak RandomForest kullanılır.


DATASETS = {
    # İki veri setinin kısa isimleri ve GitHub raw feature dosyaları burada tanımlanır.
    "ERa_BLA_assay": {
        "short_name": "ERα BLA",
        "model_prefix": "model_ERa_BLA",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_BLA_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    },
    "ERa_LUC_VM7_assay": {
        "short_name": "ERα LUC VM7",
        "model_prefix": "model_ERa_LUC_VM7",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_LUC_VM7_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
    },
}

RANDOM_STATE = 42
# Train/test ayrımı ve modeller için sabit random seed belirlenir.
TEST_SIZE = 0.20
# Test set oranı %20 olarak belirlenir.

OUTPUT_ROOT = Path("molfest_outputs")
# Bu scriptin çıktılarının kaydedileceği ana klasör belirlenir.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.
print("Feature CSV klasörü:", "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store")
# Hazır feature dosyalarının okunacağı GitHub raw klasörü gösterilir.

## 3. Yardımcı fonksiyonlar

Bu fonksiyonlar notebook içinde görünür durumdadır ve sadece bu scriptte ihtiyaç duyulan işlemleri kapsar.

In [ ]:
def show_table(df, n=50, title=None):
    """Sonuç tablosunu Colab'da tablo olarak, terminalde metin olarak gösterir."""
    if title:
        # Tablo başlığı varsa önce başlık yazdırılır.
        print("\n" + title)
    try:
        # Colab/Jupyter ortamında display daha okunabilir tablo üretir.
        display(df.head(n))
    except NameError:
        # Terminalde display yoksa tablo metin olarak yazdırılır.
        print(df.head(n).to_string(index=False))


def load_feature_table(dataset_key):
    """Seçilen veri setinin hazır feature CSV dosyasını GitHub raw linkinden okur."""
    info = DATASETS[dataset_key]
    # Veri setine ait URL ve kısa isim bilgisi alınır.
    df = pd.read_csv(info["feature_url"])
    # Feature CSV doğrudan GitHub raw linkinden okunur.
    print(f"\n{dataset_key} okundu: {df.shape[0]} satır, {df.shape[1]} kolon")
    # Okunan tablonun boyutu ekrana yazdırılır.
    return df
    # Okunan tablo modelleme adımlarına gönderilir.


def feature_columns(df, feature_set):
    """İstenen feature ailesine ait kolonları seçer."""
    prefix_map = {
        # Feature set adları CSV kolon prefixleriyle eşleştirilir.
        "morgan": ["Morgan_"],
        "maccs": ["MACCS_"],
        "rdkit": ["RDKit_"],
        "avalon": ["Avalon_"],
        "maccs_morgan": ["MACCS_", "Morgan_"],
        "maccs_rdkit": ["MACCS_", "RDKit_"],
        "morgan_rdkit": ["Morgan_", "RDKit_"],
        "all_available": ["Morgan_", "MACCS_", "RDKit_", "Avalon_"],
    }
    prefixes = prefix_map[feature_set]
    # İstenen feature setinin kolon prefixleri alınır.
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    # Prefix ile eşleşen feature kolonları seçilir.
    if not cols:
        # Hiç kolon bulunmazsa erken ve anlaşılır hata verilir.
        raise ValueError(f"{feature_set} için feature kolonu bulunamadı.")
    return cols
    # Seçilen feature kolon isimleri döndürülür.



def available_feature_sets(df):
    """Feature CSV içinde gerçekten bulunan feature setlerini listeler."""
    sets = [
        "morgan",
        "maccs",
        "rdkit",
        "maccs_morgan",
        "maccs_rdkit",
        "morgan_rdkit",
        "all_available",
    ]
    # Temel feature set kombinasyonları tanımlanır.

    if any(c.startswith("Avalon_") for c in df.columns):
        # CSV içinde Avalon kolonları varsa Avalon ayrı bir feature set olarak eklenir.
        sets.insert(3, "avalon")

    return sets
    # Bu liste feature ablation döngüsünde sırayla denenecek feature setlerini verir.

def make_xy(df, cols):
    """Feature kolonlarından X matrisi ve Target kolonundan y vektörü üretir."""
    X = (
        df[cols]
        # Sadece seçilen feature kolonları alınır.
        .apply(pd.to_numeric, errors="coerce")
        # Sayısal olmayan değerler güvenli şekilde NaN yapılır.
        .replace([np.inf, -np.inf], np.nan)
        # Sonsuz değerler NaN'a çevrilir.
        .fillna(0.0)
        # NaN değerler model eğitimi bozulmasın diye 0 yapılır.
        .to_numpy(dtype=np.float32)
        # sklearn modelleri için numpy matrise çevrilir.
    )
    y = df["Target"].astype(int).to_numpy()
    # Binary target kolonu integer numpy vektörüne çevrilir.
    return X, y
    # Modelleme için X ve y döndürülür.


def split_data(df, cols):
    """Sınıf oranını koruyarak train/test ayrımı yapar."""
    X, y = make_xy(df, cols)
    # Seçilen featurelardan X ve y hazırlanır.
    return train_test_split(X, y, df, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    # Stratify ile sınıf oranı train/test içinde korunur.


def class1_score(model, X):
    """Modelden class 1 için skor veya olasılık üretir."""
    if hasattr(model, "predict_proba"):
        # Model olasılık üretebiliyorsa class 1 olasılığı alınır.
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        # Olasılık yoksa karar fonksiyonu skoru kullanılır.
        return model.decision_function(X)
    return model.predict(X).astype(float)
    # Son seçenek olarak sınıf tahmini skor gibi kullanılır.


def metric_dict(y_true, y_pred, y_score):
    """Test tahminlerinden temel sınıflandırma metriklerini hesaplar."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # Confusion matrix değerleri ayrı ayrı alınır.
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    # Specificity = TN / (TN + FP) olarak hesaplanır.
    return {
        "ROC": roc_auc_score(y_true, y_score),
        # ROC-AUC, pozitif ve negatif sınıf ayrım gücünü ölçer.
        "AP": average_precision_score(y_true, y_score),
        # AP, precision-recall eğrisi altındaki ortalama performanstır.
        "F1": f1_score(y_true, y_pred, zero_division=0),
        # F1, precision ve recall dengesini özetler.
        "Accuracy": accuracy_score(y_true, y_pred),
        # Accuracy, toplam doğru sınıflandırma oranıdır.
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        # Balanced accuracy, sınıf dengesizliğinde daha adil bir doğruluk ölçüsüdür.
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        # Recall = TP / (TP + FN) olarak hesaplanır.
        "Specificity": specificity,
        # Specificity = TN / (TN + FP) olarak hesaplanır.
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        # Precision = TP / (TP + FP) olarak hesaplanır.
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def plot_metric_bar(df, label_col, metric, title, out_file=None, top_n=None):
    """Performans sonuçlarını yatay bar chart olarak çizer."""
    plot_df = df.sort_values(metric, ascending=False).copy()
    # En iyi sonuçlar üstte olacak şekilde metrik değerine göre sıralanır.
    if top_n is not None:
        # Çok uzun tablolar için ilk n sonuç çizilebilir.
        plot_df = plot_df.head(top_n)
    plot_df = plot_df.sort_values(metric, ascending=True)
    # Yatay bar chartta en iyi değer üstte görünsün diye çizim öncesi ters sıralanır.

    plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
    # Grafik boyutu satır sayısına göre ayarlanır.
    plt.barh(plot_df[label_col].astype(str), plot_df[metric])
    # Her model/feature set için metrik değeri yatay bar olarak çizilir.
    plt.xlabel(metric)
    # X eksenine hangi metrik çizildiği yazılır.
    plt.title(title)
    # Grafiğe açıklayıcı başlık eklenir.
    plt.tight_layout()
    # Etiketlerin kesilmemesi için yerleşim düzenlenir.
    if out_file:
        # Dosya yolu verilmişse grafik kaydedilir.
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        # Kayıt klasörü yoksa oluşturulur.
        plt.savefig(out_file, dpi=300, bbox_inches="tight")
        # Grafik yüksek çözünürlüklü PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.


def add_gain(current_df, previous_df, current_name, previous_name):
    """Bir önceki adıma göre ROC/AP/F1 farkını ve yüzde değişimi hesaplar."""
    rows = []
    # Hesaplanan karşılaştırma satırları burada biriktirilir.
    for _, current in current_df.iterrows():
        # Mevcut adımın her veri seti sonucu tek tek dolaşılır.
        dataset = current["Dataset"]
        # Karşılaştırılacak veri seti adı alınır.
        prev = previous_df[previous_df["Dataset"] == dataset].sort_values("ROC", ascending=False).iloc[0]
        # Aynı veri setinin önceki adımdaki en iyi sonucu alınır.
        row = current.to_dict()
        # Mevcut sonuç satırı sözlüğe çevrilir.
        row["CurrentStep"] = current_name
        # Mevcut adım adı eklenir.
        row["ComparedWith"] = previous_name
        # Karşılaştırılan önceki adım adı eklenir.
        row["Previous_ROC"] = prev["ROC"]
        # Önceki ROC değeri tabloya eklenir.
        row["ROC_Delta"] = current["ROC"] - prev["ROC"]
        # Mutlak ROC farkı hesaplanır.
        row["ROC_Gain_%"] = 100 * (current["ROC"] - prev["ROC"]) / abs(prev["ROC"]) if abs(prev["ROC"]) > 1e-12 else np.nan
        # ROC yüzde değişimi hesaplanır.
        row["Previous_AP"] = prev["AP"]
        # Önceki AP değeri tabloya eklenir.
        row["AP_Delta"] = current["AP"] - prev["AP"]
        # AP farkı hesaplanır.
        row["AP_Gain_%"] = 100 * (current["AP"] - prev["AP"]) / abs(prev["AP"]) if abs(prev["AP"]) > 1e-12 else np.nan
        # AP yüzde değişimi hesaplanır.
        row["Previous_F1"] = prev["F1"]
        # Önceki F1 değeri tabloya eklenir.
        row["F1_Delta"] = current["F1"] - prev["F1"]
        # F1 farkı hesaplanır.
        row["F1_Gain_%"] = 100 * (current["F1"] - prev["F1"]) / abs(prev["F1"]) if abs(prev["F1"]) > 1e-12 else np.nan
        # F1 yüzde değişimi hesaplanır.
        rows.append(row)
        # Karşılaştırma satırı listeye eklenir.
    return pd.DataFrame(rows)
    # Tüm karşılaştırmalar tablo olarak döndürülür.


def make_random_forest():
    """Baseline ve gatekeeper için RandomForest modeli kurar."""
    return RandomForestClassifier(
        n_estimators=300,
        # 300 ağaç, eğitim süresi ve stabilite arasında dengeli bir seçimdir.
        max_features="sqrt",
        # Her bölünmede featureların karekökü kadar aday denenir.
        class_weight="balanced_subsample",
        # Sınıf dengesizliğini her bootstrap örneğinde dengelemeye çalışır.
        random_state=RANDOM_STATE,
        # Tekrarlanabilirlik sağlar.
        n_jobs=-1,
        # Mevcut tüm işlemciler kullanılır.
    )

## 4. Baseline ve feature ablation

Önce RF + Morgan baseline hesaplanır. Sonra RandomForest sabit tutularak farklı feature setleri karşılaştırılır.

In [ ]:
lesson_out = OUTPUT_ROOT / "03_rf_feature_ablation"
# Feature ablation çıktıları için klasör belirlenir.
lesson_out.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

baseline_rows = []
# Baseline sonuçları bu listede tutulur.
ablation_rows = []
# Tüm feature ablation sonuçları bu listede tutulur.
best_rows = []
# Her veri seti için en iyi feature set sonucu bu listede tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    baseline_features = feature_columns(df, "morgan")
    # Baseline için Morgan featureları seçilir.
    X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, baseline_features)
    # Morgan featurelarıyla train/test ayrımı yapılır.

    baseline_model = make_random_forest()
    # Baseline RandomForest modeli kurulur.
    baseline_model.fit(X_train, y_train)
    # Baseline model train set üzerinde eğitilir.

    baseline_pred = baseline_model.predict(X_test)
    # Baseline model test tahmini üretir.
    baseline_score = class1_score(baseline_model, X_test)
    # ROC/AP için baseline class 1 skoru alınır.

    baseline_metrics = metric_dict(y_test, baseline_pred, baseline_score)
    # Baseline metrikleri hesaplanır.
    baseline_metrics.update({
        "Dataset": dataset_key,
        "Step": "02_baseline",
        "ModelType": "RandomForest",
        "FeatureSet": "morgan",
        "FeatureCount": len(baseline_features),
    })
    # Baseline sonuç satırına metadata eklenir.
    baseline_rows.append(baseline_metrics)
    # Baseline sonucu listeye eklenir.

    dataset_rows = []
    # Bu veri setinin feature ablation sonuçları burada tutulur.

    for feature_set in available_feature_sets(df):
        # CSV içinde mevcut olan feature setleri sırayla denenir.
        selected_features = feature_columns(df, feature_set)
        # İlgili feature setin kolonları seçilir.
        X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, selected_features)
        # Seçilen feature setiyle train/test ayrımı yapılır.

        model = make_random_forest()
        # Karşılaştırmanın adil olması için model yine RandomForest olarak tutulur.
        model.fit(X_train, y_train)
        # Model train set üzerinde eğitilir.

        y_pred = model.predict(X_test)
        # Test seti için sınıf tahmini alınır.
        y_score = class1_score(model, X_test)
        # ROC/AP için class 1 skoru alınır.

        metrics = metric_dict(y_test, y_pred, y_score)
        # Test performans metrikleri hesaplanır.
        metrics.update({
            "Dataset": dataset_key,
            "Step": "03_feature_ablation",
            "ModelType": "RandomForest",
            "FeatureSet": feature_set,
            "FeatureCount": len(selected_features),
        })
        # Sonuç satırına feature set bilgileri eklenir.

        dataset_rows.append(metrics)
        # Bu veri setinin sonuç listesine eklenir.
        ablation_rows.append(metrics)
        # Genel ablation sonuç listesine eklenir.

    dataset_df = pd.DataFrame(dataset_rows).sort_values("ROC", ascending=False)
    # Bu veri setinin sonuçları ROC değerine göre sıralanır.
    dataset_df.to_csv(lesson_out / f"{dataset_key}_feature_ablation.csv", index=False)
    # Veri setine ait ablation sonuçları CSV olarak kaydedilir.

    plot_metric_bar(
        dataset_df,
        label_col="FeatureSet",
        metric="ROC",
        title=f"{dataset_key} — feature set ROC karşılaştırması",
        out_file=lesson_out / f"{dataset_key}_feature_ablation_roc.png",
    )
    # Feature set ROC değerleri bar chart olarak çizilir.

    show_table(dataset_df, title=f"{dataset_key} — feature ablation tablosu")
    # Feature ablation sonuç tablosu ekranda gösterilir.
    best_rows.append(dataset_df.iloc[0].to_dict())
    # En iyi feature set sonucu sonraki karşılaştırma için saklanır.

baseline_df = pd.DataFrame(baseline_rows)
# Baseline sonuçları tabloya çevrilir.
best_feature_df = pd.DataFrame(best_rows)
# En iyi feature set sonuçları tabloya çevrilir.
ablation_df = pd.DataFrame(ablation_rows)
# Tüm ablation sonuçları tabloya çevrilir.

baseline_df.to_csv(lesson_out / "03_recomputed_baseline.csv", index=False)
# Baseline sonuçları CSV olarak kaydedilir.
ablation_df.to_csv(lesson_out / "03_feature_ablation_all.csv", index=False)
# Tüm ablation sonuçları CSV olarak kaydedilir.
best_feature_df.to_csv(lesson_out / "03_best_feature_set_per_dataset.csv", index=False)
# Veri seti bazında en iyi feature set sonuçları CSV olarak kaydedilir.

gain_df = add_gain(best_feature_df, baseline_df, "03_Feature_Ablation", "02_RF_Morgan_Baseline")
# En iyi feature set sonucu baseline ile karşılaştırılır.
gain_df.to_csv(lesson_out / "03_gain_vs_baseline.csv", index=False)
# Kazanç tablosu CSV olarak kaydedilir.

# -------------------------------------------------------------------------
# Morgan reference ve en iyi feature set karşılaştırmasını ayrı tablolarla verelim.
# Burada amaç, her veri seti için referans satırını ve en iyi adayı aynı tabloda görmektir.
# Böylece sadece kazanç değerini değil, referans performansın kendisini de açıkça görürüz.
# -------------------------------------------------------------------------

comparison_tables = []
# İki veri setinin reference-vs-best tabloları bu listede biriktirilir.

for dataset_key in DATASETS:
    # Her classification veri seti için ayrı tablo üretilecek.
    reference_row = baseline_df[baseline_df["Dataset"] == dataset_key].iloc[0]
    # Reference satırı RF + Morgan baseline sonucudur.
    best_row = best_feature_df[best_feature_df["Dataset"] == dataset_key].iloc[0]
    # Best satırı feature ablation sonunda ROC'a göre seçilen en iyi feature set sonucudur.

    reference_roc = reference_row["ROC"]
    # Morgan reference ROC değeri alınır.
    reference_ap = reference_row["AP"]
    # Morgan reference AP değeri alınır.
    reference_f1 = reference_row["F1"]
    # Morgan reference F1 değeri alınır.

    comparison_df = pd.DataFrame([
        {
            "Dataset": dataset_key,
            "Comparison": "Reference",
            "ModelType": "RandomForest",
            "FeatureSet": reference_row["FeatureSet"],
            "FeatureCount": reference_row["FeatureCount"],
            "ROC": reference_row["ROC"],
            "AP": reference_row["AP"],
            "F1": reference_row["F1"],
            "ROC_Delta_vs_Morgan": 0.0,
            "ROC_Gain_%": 0.0,
            "AP_Delta_vs_Morgan": 0.0,
            "AP_Gain_%": 0.0,
            "F1_Delta_vs_Morgan": 0.0,
            "F1_Gain_%": 0.0,
        },
        # İlk satır referanstır: RF + Morgan. Bu yüzden bütün gain değerleri 0'dır.
        {
            "Dataset": dataset_key,
            "Comparison": "Best feature set",
            "ModelType": "RandomForest",
            "FeatureSet": best_row["FeatureSet"],
            "FeatureCount": best_row["FeatureCount"],
            "ROC": best_row["ROC"],
            "AP": best_row["AP"],
            "F1": best_row["F1"],
            "ROC_Delta_vs_Morgan": best_row["ROC"] - reference_roc,
            "ROC_Gain_%": 100 * (best_row["ROC"] - reference_roc) / abs(reference_roc) if abs(reference_roc) > 1e-12 else np.nan,
            "AP_Delta_vs_Morgan": best_row["AP"] - reference_ap,
            "AP_Gain_%": 100 * (best_row["AP"] - reference_ap) / abs(reference_ap) if abs(reference_ap) > 1e-12 else np.nan,
            "F1_Delta_vs_Morgan": best_row["F1"] - reference_f1,
            "F1_Gain_%": 100 * (best_row["F1"] - reference_f1) / abs(reference_f1) if abs(reference_f1) > 1e-12 else np.nan,
        },
        # İkinci satır en iyi feature set sonucudur ve Morgan reference'a göre gain değerlerini içerir.
    ])

    comparison_csv = lesson_out / f"{dataset_key}_morgan_reference_vs_best_feature_set.csv"
    # Bu veri setine ait iki satırlık karşılaştırma tablosunun CSV yolu belirlenir.
    comparison_df.to_csv(comparison_csv, index=False)
    # Reference-vs-best tablosu CSV olarak kaydedilir.

    show_table(
        comparison_df,
        title=f"{dataset_key} — Morgan reference vs en iyi feature set"
    )
    # Bu veri seti için iki satırlık reference-vs-best tablosu ekranda gösterilir.

    comparison_tables.append(comparison_df)
    # Tablo genel özet için listeye eklenir.

reference_vs_best_df = pd.concat(comparison_tables, ignore_index=True)
# İki veri setinin reference-vs-best tabloları tek genel tabloda birleştirilir.
reference_vs_best_df.to_csv(lesson_out / "03_morgan_reference_vs_best_feature_set_all.csv", index=False)
# Genel reference-vs-best tablosu CSV olarak kaydedilir.